# 01 — Clifford Basics and Crank Words

This notebook talks to the real Crankl C library from Python.

**In plain terms:** Crankl stores each compressed block as a short list of 8 numbers (a “multivector”). Those 8 slots are labeled:

`1, e1, e2, e3, e12, e23, e13, e123`

You can think of them as: one overall scale, three direction pieces, three plane pieces, and one “volume” piece.

We will walk through:

1. Multiplying basis pieces (e.g. `e1 * e1 = 1`, `e1 * e2 = e12`)
2. Reversing a piece (flips the sign of some slots)
3. Packing those 8 numbers into a 64-bit crank word, then unpacking them
4. Turning a crank word into an 8×8 matrix (“decrank”)
5. Measuring how similar two crank words are (“resonance”)

Same functions as in `include/crankl/clifford.h` and `include/crankl/crank.h`.

In [ ]:
from pathlib import Path
import sys
import numpy as np

notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(notebook_dir)) if str(notebook_dir) not in sys.path else None

from crankl_demo import CranklAPI, Multivector, print_matrix

np.set_printoptions(precision=3, suppress=True)
api = CranklAPI()

# Construct exact Cl(3) basis elements using the public multivector struct.
one = Multivector.create(scalar=1.0)
e1 = Multivector.create(vector=(1.0, 0.0, 0.0))
e2 = Multivector.create(vector=(0.0, 1.0, 0.0))
e12 = Multivector.create(bivector=(1.0, 0.0, 0.0))

basis_labels = ["1", "e1", "e2", "e3", "e12", "e23", "e13", "e123"]
print("Coefficient order:", basis_labels)
print("e1 coefficients:", e1.as_array())

## Multiply and reverse

Multiplying two of these 8-number objects is like combining directions:

- same direction twice → you get a plain number (`e1 * e1 = 1`)
- two different directions → you get the “plane” between them (`e1 * e2 = e12`)

**Reversion** flips the sign of the plane and volume slots, and leaves the scale and direction slots alone. Useful as a basic algebra check.

In [ ]:
e1_squared = api.clifford_product(e1, e1)
e1_times_e2 = api.clifford_product(e1, e2)
e12_squared = api.clifford_product(e12, e12)
reversed_e12 = api.reversion(e12)

print("e1 * e1  =", e1_squared.as_array(), "  (scalar 1 expected)")
print("e1 * e2  =", e1_times_e2.as_array(), "  (e12 expected)")
print("e12 * e12=", e12_squared.as_array(), "  (scalar -1 expected)")
print("reverse(e12) =", reversed_e12.as_array(), "  (-e12 expected)")

## Encode, decode, rebuild a grid, measure similarity

Encoding squeezes the 8 continuous numbers into a single 64-bit word. Some detail is lost on purpose (like JPEG for images) — so decoding gives a **rounded** version of what you put in, not always the exact floats.

- **Decrank** expands a crank word back into an 8×8 matrix you can print and compare.
- **Resonance** is a similarity score between two words: ~1 means “almost the same,” ~0 means “very different / unrelated.”

In [ ]:
continuous = Multivector.create(
    scalar=0.5,
    vector=(0.8, 0.1, -0.7),
    bivector=(0.6, 0.0, -0.9),
    pseudoscalar=0.4,
)
word = api.encode(continuous, depth=3)
decoded, depth = api.decode(word)

print(f"Encoded crank word: 0x{word:016x}")
print("Continuous input:  ", continuous.as_array())
print("Decoded/quantized: ", decoded.as_array())
print("Decoded depth:     ", depth)

print(f"\nRes(e1, e1) = {api.resonance(api.encode(e1), api.encode(e1)):.6f}")
print(f"Res(e1, e2) = {api.resonance(api.encode(e1), api.encode(e2)):.6f}")
print_matrix("Decranked 8x8 operator", api.decrank(word))

## Ternary digits (trits) under the hood

Many of the geometric slots are stored as **trits**: only `-1`, `0`, or `+1`. Each trit is packed into two bits.

You normally never call this yourself — packing handles it — but the next cell shows the tiny encode/decode round trip.

In [ ]:
for trit in (-1, 0, 1):
    encoded = api.encode_trit(trit)
    decoded = api.decode_trit(encoded)
    print(f"trit={trit:+d} -> bits=0b{encoded:02b} -> decoded={decoded:+d}")
    assert decoded == trit